In [1]:
#Imports
%load_ext autoreload
%autoreload 2
import networks, data_loaders, image_video_tools, training_functions
import cv2
import importlib
import torch
import os
import testing_and_metrics, testing_and_metrics_multimodal
import timeseries_tools
import pandas as pd
from torch.utils.tensorboard import SummaryWriter
from training_functions import kfold_train_multimodal
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("- Imports completed successfully -")

/home/ws/ugoby/master_thesis/masterthesis/image_video_tools.py:81: UserWarning: Argument(s) 'var_limit, mean' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(8.0, 40.0), mean=0, per_channel=True, p=0.3),
/home/ws/ugoby/master_thesis/masterthesis/image_video_tools.py:87: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=30, quality_upper=65, p=0.75),
/home/ws/ugoby/master_thesis/masterthesis/image_video_tools.py:88: UserWarning: Argument(s) 'num_flare_circles_lower, num_flare_circles_upper, angle_lower, angle_upper' are not valid for transform RandomSunFlare
  A.RandomSunFlare(


- Imports completed successfully -


DOWNLOAD KAGGLE

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jefmenegazzo/pvs-passive-vehicular-sensors-datasets")

print("Path to dataset files:", path)

100%|██████████| 41.4G/41.4G [19:48<00:00, 37.4MB/s]  

Extracting files...


Path to dataset files: /home/ws/ugoby/.cache/kagglehub/datasets/jefmenegazzo/pvs-passive-vehicular-sensors-datasets/versions/2


In [1]:
from torch.utils.tensorboard import SummaryWriter

K-FOLD CROSS VALIDATION

In [ ]:
# Example usage with your existing experiment block (multimodal):
param_block = {
    "dataset_type": "multimodal",
    "image_folders": [
        "/home/ws/ugoby/master_thesis/data/pvs9_video+",
        "/home/ws/ugoby/master_thesis/data/pvs9_mfcc++_timewarp",
    ],
    "backbone_configs": [
        {"network": networks.ConvNeXtBaseClassifier, "path": "/home/ws/ugoby/master_thesis/models/thesis/pvs9_video+.pt", "nr_classes": 3},
        {"network": networks.ConvNeXtBaseClassifier, "path": "/home/ws/ugoby/master_thesis/models/thesis/pvs9_mfcc++_timewarp.pt", "nr_classes": 3},
    ],
    "dataset_builder": None,  # or provide your callable to return (dataset, labels, class_to_idx)
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": lambda bc, dev: (lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3))(*networks.build_backbones(bc, torch.device("cuda" if torch.cuda.is_available() else "cpu"))),
    "resize": (224, 224),
    "epochs": 50,
    "model_save_path": "/home/ws/ugoby/master_thesis/models/kfold/MM_pvs9_video+_pvs9_mfcc++.pt",
    "num_classes": 3,
    "early_stopping_patience": 10,
    "min_delta": 0.001,
    "log_dir_base": "tensorboard/runs/kfold/MM_kfold_pvs9_video+_pvs9_mfccpp"
}

# Run 5-fold CV
fold_rows, summary_txt = kfold_train_multimodal(param_block, K=5, seed=42, summary_txt_path="/home/ws/ugoby/master_thesis/test_results/kfold/MM_pvs9_video+_pvs9_mfcc++_kfold_summary.txt")
print("Summary TXT:", summary_txt)


Total samples: 15687, num_classes: 3

===== Fold 1/5 =====


/home/ws/ugoby/master_thesis/masterthesis/training_functions.py:525: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None



Epoch 1/50


Training: 100%|██████████| 393/393 [00:20<00:00, 19.27it/s, acc=99.4, loss=0.00159]


Train Loss: 0.2134, Train Acc: 99.35%
Val   Loss: 0.0055, Val   Acc: 99.84%
✔ Best model updated and saved to /home/ws/ugoby/master_thesis/models/kfold/MM_pvs9_video+_pvs9_mfcc++__fold1of5.pt

Epoch 2/50


...

Training: 100%|██████████| 393/393 [00:16<00:00, 23.93it/s, acc=99.9, loss=0.00104] 


Train Loss: 0.0020, Train Acc: 99.94%
Val   Loss: 0.0014, Val   Acc: 99.90%
⚠ No improvement for 4 epoch(s).

Epoch 6/50


Training: 100%|██████████| 393/393 [00:17<00:00, 23.08it/s, acc=99.9, loss=3.82e-5] 


Train Loss: 0.0022, Train Acc: 99.92%
Val   Loss: 0.0015, Val   Acc: 99.90%
⚠ No improvement for 5 epoch(s).

Epoch 7/50


Training:   9%|▉         | 37/393 [00:01<00:14, 24.08it/s, acc=99.9, loss=6.92e-5] 

HYPERPARAMETEROPTIMIZATION

In [ ]:
#Single Model
parameters = [
    {
    "image_folder": r"D:\Uni\Coding\PVS 1\extracted_images_1fps",
    "csv_file": r"data\synchronized_dataset_cutting_AB\high_images\synchronized_dataset_AB_high.csv",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": networks.ResNet18Classifier,
    "resize": (224, 224),
    "epochs": 10,
    "writer": SummaryWriter(log_dir="tensorboard/runs/multi_training_session_R18_high_images"),
    "model_save_path": r"models\preparing_data_models\best_model_R18_high.pt",
    "data_type": "classic",

    # Early stopping additions:
    "early_stopping_patience": 5,
    "min_delta": 0.001
}
]

search_space = {
    "learning_rate": [1e-4, 1e-2],
    "resize": [(224, 224), (64, 64)],
    "model_class": [networks.ResNet18Classifier, networks.ConvNeXtTinyClassifier],
}


#training_functions.grid_search_train(parameters[0], search_space, "grid_search_experiment")
#training_functions.random_search_train(parameters[0], search_space, 3, "random_search_experiment")

In [ ]:
### Multimodal Grid Search ###
# The number of backbone configs determine the number of feature extraction backbones to be used
# The number of image folders should match the number of backbone configs
# The files inside the image folders corresponding to the same sample must have the same name 
# The backbones that are loaded should be pretrained on the same dataset as the images in the folders

model_path =r"models\noise_to_image\evaluating_noiseimages_R50_mfcc_batch1.pt"
model_path =r"models\noise_to_image\evaluating_noiseimages_R50_mfcc_batch1.pt"

#No early stopping parameter
parameters = [
    {
        "backbone_configs": [
            {"network": networks.ResNet50Classifier, "path": model_path, "nr_classes": 3},
            {"network": networks.ConvNeXtBaseClassifier, "path": r"models\noise_to_image\evaluating_noiseimages_CNb_mfcc_batch1.pt", "nr_classes": 3},
            {"network": networks.ViT_L_16_Classifier, "path": r"models\noise_to_image\evaluating_noiseimages_VITl16_mfcc_batch1.pt", "nr_classes": 3},
        ],
        "image_folders": [
            r"data\image_data\mfcc_40_tiny",
            r"data\image_data\mfcc_40_tiny",
            r"data\image_data\mfcc_40_tiny",
        ],
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": lambda bc, dev: (
            lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
        )(*networks.build_backbones(bc, dev)),
        "resize": (224, 224),
        "epochs": 5,
        "writer": SummaryWriter(log_dir="tensorboard/runs/test_multi_modal"),
        "model_save_path": r"models/preparing_data_models/test_multi_modal_session_R50_high_images.pt",
        "data_type": "multimodal"
    }
]

search_space = {
    "learning_rate": [1e-4, 1e-3, 1e-2],
    "resize": [(224, 224)]
}

#training_functions.train_multiple_runs(parameters)
training_functions.grid_search_train(parameters[0], search_space, "26_06_multimodal_data_experiment")

MULTIMODAL TRAINING AREA

In [ ]:
### Multimodal Training ###
# The number of backbone configs determine the number of feature extraction backbones to be used
# The number of image folders should match the number of backbone configs
# The files inside the image folders corresponding to the same sample must have the same name 
# The backbones that are loaded should be pretrained on the same dataset as the images in the folders




#### MM_pvs1_video_pvs1_mfcc
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_video.pt"
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pvs1_mfcc.pt"


parameters = [
    {
    "backbone_configs": [
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},

    ],
    "image_folders": [
        r"/home/ws/ugoby/master_thesis/data/pvs1_video",
        r"/home/ws/ugoby/master_thesis/data/pvs1_mfcc",
    ],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": lambda bc, dev: (
        lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
    )(*networks.build_backbones(bc, dev)),
    "resize": (224, 224),
    "epochs": 50,
    "writer": SummaryWriter(log_dir="tensorboard/runs/MM_pvs1_video_pvs1_mfcc"),
    "model_save_path": r"models/preparing_data_models/MM_pvs1_video_pvs1_mfcc.pt",
    "data_type": "multimodal",
    "num_classes": 3,
    
    # Early stopping additions:
    "early_stopping_patience": 10,
    "min_delta": 0.001
},
]

training_functions.train_multiple_runs(parameters)



#### MM_pvs9_video_pvs9_mfcc
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs9_video.pt"
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pvs9_mfcc.pt"


parameters = [
    {
    "backbone_configs": [
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},

    ],
    "image_folders": [
        r"/home/ws/ugoby/master_thesis/data/pvs9_video",
        r"/home/ws/ugoby/master_thesis/data/pvs9_mfcc",
    ],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": lambda bc, dev: (
        lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
    )(*networks.build_backbones(bc, dev)),
    "resize": (224, 224),
    "epochs": 50,
    "writer": SummaryWriter(log_dir="tensorboard/runs/MM_pvs9_video_pvs9_mfcc"),
    "model_save_path": r"models/preparing_data_models/MM_pvs9_video_pvs9_mfcc.pt",
    "data_type": "multimodal",
    "num_classes": 3,
    
    # Early stopping additions:
    "early_stopping_patience": 10,
    "min_delta": 0.001
},
]

training_functions.train_multiple_runs(parameters)


#### MM_pvs1_video_pvs1_mfcc++
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_video.pt"
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_mfcc++.pt"


parameters = [
    {
    "backbone_configs": [
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},

    ],
    "image_folders": [
        r"/home/ws/ugoby/master_thesis/data/pvs1_video",
        r"/home/ws/ugoby/master_thesis/data/pvs1_mfcc++",
    ],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": lambda bc, dev: (
        lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
    )(*networks.build_backbones(bc, dev)),
    "resize": (224, 224),
    "epochs": 50,
    "writer": SummaryWriter(log_dir="tensorboard/runs/MM_pvs1_video_pvs1_mfcc++"),
    "model_save_path": r"models/preparing_data_models/MM_pvs1_video_pvs1_mfcc++.pt",
    "data_type": "multimodal",
    "num_classes": 3,
    
    # Early stopping additions:
    "early_stopping_patience": 10,
    "min_delta": 0.001
},
]

training_functions.train_multiple_runs(parameters)



#### MM_pvs9_video++_pvs9_mfcc
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs9_video++.pt"
model_path =r"/home/ws/ugoby/master_thesis/models/single_backbones/pvs9_mfcc.pt"


parameters = [
    {
    "backbone_configs": [
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},
        {"network": networks.ConvNeXtBaseClassifier, "path": model_path, "nr_classes": 3},

    ],
    "image_folders": [
        r"/home/ws/ugoby/master_thesis/data/pvs9_video++",
        r"/home/ws/ugoby/master_thesis/data/pvs9_mfcc",
    ],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": 1e-4,
    "batch_size": 32,
    "model_class": lambda bc, dev: (
        lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
    )(*networks.build_backbones(bc, dev)),
    "resize": (224, 224),
    "epochs": 50,
    "writer": SummaryWriter(log_dir="tensorboard/runs/MM_pvs9_video++_pvs9_mfcc"),
    "model_save_path": r"models/preparing_data_models/MM_pvs9_video++_pvs9_mfcc.pt",
    "data_type": "multimodal",
    "num_classes": 3,
    
    # Early stopping additions:
    "early_stopping_patience": 10,
    "min_delta": 0.001
},
]

training_functions.train_multiple_runs(parameters)


----- Starting training run 1 -----


m:\Coding\Master\masterthesis\training_functions.py:506: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None



Epoch 1/30


Training: 100%|██████████| 1548/1548 [06:16<00:00,  4.12it/s, acc=72.3, loss=0.468]


Train Loss: 0.6947, Train Acc: 72.30%
Val   Loss: 0.4249, Val   Acc: 86.87%
✔ Best model updated and saved to models/preparing_data_models/Night_MM_video_mfcc_acc.pt

Epoch 2/30


...

Train Loss: 0.0949, Train Acc: 96.91%
Val   Loss: 0.0786, Val   Acc: 97.54%
⚠ No improvement for 4 epoch(s).

Epoch 20/30


Training: 100%|██████████| 1548/1548 [08:18<00:00,  3.11it/s, acc=97.2, loss=0.124]  


Train Loss: 0.0886, Train Acc: 97.20%
Val   Loss: 0.0789, Val   Acc: 97.37%
⚠ No improvement for 5 epoch(s).

Epoch 21/30


Training:   7%|▋         | 106/1548 [00:28<06:31,  3.68it/s, acc=97.1, loss=0.0555]


KeyboardInterrupt: 

In [ ]:
## Special Multimodal Training ##

video_b_path = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs9_video++.pt"
mfcc_b_path  = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_mfcc++.pt"

parameters = [
    {
        "backbone_configs": [
            {"network": networks.ConvNeXtBaseClassifier, "path": video_b_path, "nr_classes": 3},  # video backbone
            {"network": networks.ConvNeXtBaseClassifier, "path": mfcc_b_path,  "nr_classes": 3},  # mfcc backbone
        ],

        # Explicit folder pairs instead of image_folders
        "folder_pairs": [
            (r"/home/ws/ugoby/master_thesis/data/pvs1_video",   r"/home/ws/ugoby/master_thesis/data/pvs1_mfcc++"),
            (r"/home/ws/ugoby/master_thesis/data/pvs9_video++", r"/home/ws/ugoby/master_thesis/data/pvs9_mfcc"),
        ],

        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,

        "model_class": lambda bc, dev: (
            lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
        )(*networks.build_backbones(bc, dev)),

        "resize": (224, 224),
        "epochs": 50,
        "writer": SummaryWriter(log_dir="/home/ws/ugoby/master_thesis/tensorboard/runs/multimodal/MMspecial_2b_pvs1++pvs9++"),
        "model_save_path": r"/home/ws/ugoby/master_thesis/models/multimodal/MMspecial_2b_pvs1++pvs9++.pt",

        # NEW: special loader branch, no duplication
        "data_type": "multimodal_special",
        "duplicate": False,          # <- important: yields (video, mfcc, label)

        "num_classes": 3,
        "early_stopping_patience": 10,
        "min_delta": 0.001,
    },
]

training_functions.train_multiple_runs(parameters)



----- Starting training run 1 -----


/home/ws/ugoby/master_thesis/masterthesis/training_functions.py:508: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None



Epoch 1/50


Training:  97%|█████████▋| 1107/1142 [03:12<00:06,  5.76it/s, acc=93.1, loss=0.192] 


KeyboardInterrupt: 

In [4]:
## Special Multimodal Training (4 backbones, duplicated) ##

video_b1_path = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_video.pt"
video_b2_path = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs9_video++.pt"
mfcc_b1_path  = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs1_mfcc++.pt"
mfcc_b2_path  = r"/home/ws/ugoby/master_thesis/models/single_backbones/pretty_good/pvs9_mfcc.pt"

parameters = [
    {
        "backbone_configs": [
            {"network": networks.ConvNeXtBaseClassifier, "path": video_b1_path, "nr_classes": 3},  # video backbone #1
            {"network": networks.ConvNeXtBaseClassifier, "path": video_b2_path, "nr_classes": 3},  # video backbone #2
            {"network": networks.ConvNeXtBaseClassifier, "path": mfcc_b1_path,  "nr_classes": 3},  # mfcc backbone #1
            {"network": networks.ConvNeXtBaseClassifier, "path": mfcc_b2_path,  "nr_classes": 3},  # mfcc backbone #2
        ],

        # Explicit folder pairs instead of image_folders
        "folder_pairs": [
            (r"/home/ws/ugoby/master_thesis/data/pvs1_video",   r"/home/ws/ugoby/master_thesis/data/pvs1_mfcc++"),
            (r"/home/ws/ugoby/master_thesis/data/pvs9_video++", r"/home/ws/ugoby/master_thesis/data/pvs9_mfcc"),
        ],

        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,

        "model_class": lambda bc, dev: (
            lambda backbones, dims: networks.FlexibleFusionClassifier(backbones, dims, nr_classes=3)
        )(*networks.build_backbones(bc, dev)),

        "resize": (224, 224),
        "epochs": 50,
        "writer": SummaryWriter(log_dir="/home/ws/ugoby/master_thesis/tensorboard/runs/multimodal/MMspecial_4b_pvs1++pvs9++"),
        "model_save_path": r"/home/ws/ugoby/master_thesis/models/multimodal/MMspecial_4b_pvs1++pvs9++.pt",

        # NEW: special loader branch, duplication (default True)
        "data_type": "multimodal_special",
        "duplicate": True,           # <- yields (video, video, mfcc, mfcc, label)

        "num_classes": 3,
        "early_stopping_patience": 10,
        "min_delta": 0.001,
    },
]

training_functions.train_multiple_runs(parameters)



----- Starting training run 1 -----

Epoch 1/50


Training:  11%|█         | 126/1142 [00:26<03:37,  4.67it/s, acc=93.7, loss=0.217]


KeyboardInterrupt: 

SINGLE BACKBONE TRAINING AREA

In [2]:
parameters = [
    
    {
        "image_folder": r"data\data_base\pvs9_video",
        "csv_file": r"",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.ConvNeXtBaseClassifier,
        "resize": (224, 224),
        "epochs": 5,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\test_pvs9_video"),
        "model_save_path": r"models\single_backbone\test_pvs9_video.pt",
        "data_type": "mfcc",
        "num_classes": 3,  # 3 for normal , 6 for night

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },
    {
        "image_folder": r"data\data_base\pvs9_acc",
        "csv_file": r"",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.ConvNeXtBaseClassifier,
        "resize": (224, 224),
        "epochs": 5,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\test_pvs9_acc"),
        "model_save_path": r"models\single_backbone\test_pvs9_acc.pt",
        "data_type": "mfcc",
        "num_classes": 3,  # 3 for normal , 6 for night

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },

]


training_functions.legacy_train_multiple_runs(parameters)

NameError: name 'torch' is not defined

In [2]:
### Single Backbone Training Night ###
parameters = [
    
    {
        "image_folder": r"data\nighttime_data\mfcc_acc",
        "csv_file": r"",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.ConvNeXtBaseClassifier,
        "resize": (224, 224),
        "epochs": 5,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\Night_CNb_mfcc_acc"),
        "model_save_path": r"models\single_backbone\Night_CNb_mfcc_acc.pt",
        "data_type": "mfcc",
        "num_classes": 6,  # 3 for normal , 6 for night

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },
    {
        "image_folder": r"data\nighttime_data\raw_video",
        "csv_file": r"",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.ConvNeXtBaseClassifier,
        "resize": (224, 224),
        "epochs": 5,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\Night_CNb_video"),
        "model_save_path": r"models\single_backbone\Night_CNb_video.pt",
        "data_type": "mfcc",
        "num_classes": 6,  # 3 for normal , 6 for night

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },
]


training_functions.legacy_train_multiple_runs(parameters)


----- Starting training run 1 -----


m:\Coding\Master\masterthesis\training_functions.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None



Epoch 1/5


Training: 100%|██████████| 1548/1548 [07:00<00:00,  3.68it/s, acc=33.5, loss=1.06] 


Train Loss: 1.3120, Train Acc: 33.54%
Val   Loss: 1.1365, Val   Acc: 41.12%
✔ Best model updated and saved to models\single_backbone\Night_CNb_mfcc_acc.pt

Epoch 2/5


...

Train Loss: 0.0579, Train Acc: 97.77%
Val   Loss: 0.0673, Val   Acc: 97.98%
✔ Best model updated and saved to models\single_backbone\Night_CNb_video.pt

Epoch 4/5


Training: 100%|██████████| 1548/1548 [07:10<00:00,  3.59it/s, acc=98, loss=0.0161]    


Train Loss: 0.0512, Train Acc: 98.03%
Val   Loss: 0.0586, Val   Acc: 97.91%
⚠ No improvement for 1 epoch(s).

Epoch 5/5


Training: 100%|██████████| 1548/1548 [07:10<00:00,  3.60it/s, acc=98.3, loss=0.0217]  


Train Loss: 0.0447, Train Acc: 98.28%
Val   Loss: 0.0587, Val   Acc: 98.04%
⚠ No improvement for 2 epoch(s).

Training complete. Best val accuracy: 97.98%
----- Finished training run 2 -----



In [4]:
### Single Backbone Training ###
parameters = [
    {
        "image_folder": r"data\time_synchronized_data\mfcc_temp",
        "csv_file": r"data\time_synchronized_data\labels\X01_label.csv",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.EfficientViTClassifier,
        "resize": (224, 224),
        "epochs": 50,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\EVIT_mfcc_temp"),
        "model_save_path": r"models\single_backbone\EVIT_mfcc_temp.pt",
        "data_type": "mfcc",

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },
    {
        "image_folder": r"data\time_synchronized_data\cwt_temp",
        "csv_file": r"data\time_synchronized_data\labels\X01_label.csv",
        "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        "learning_rate": 1e-4,
        "batch_size": 32,
        "model_class": networks.EfficientViTClassifier,
        "resize": (224, 224),
        "epochs": 50,
        "writer": SummaryWriter(log_dir=r"tensorboard\runs\single_backbone\EVIT_cwt_temp"),
        "model_save_path": r"models\single_backbone\EVIT_cwt_temp.pt",
        "data_type": "cwt",

        # Early stopping additions:
        "early_stopping_patience": 10,
        "min_delta": 0.001
    },
]


training_functions.legacy_train_multiple_runs(parameters)


----- Starting training run 1 -----

Epoch 1/50


Training: 100%|██████████| 774/774 [02:16<00:00,  5.66it/s, acc=36.4, loss=1.04] 


Train Loss: 1.1229, Train Acc: 36.37%
Val   Loss: 1.0834, Val   Acc: 36.02%
✔ Best model updated and saved to models\single_backbone\EVIT_mfcc_temp.pt

Epoch 2/50


Training: 100%|██████████| 774/774 [02:16<00:00,  5.66it/s, acc=37.5, loss=1.11] 


Train Loss: 1.0903, Train Acc: 37.50%
Val   Loss: 1.0835, Val   Acc: 44.70%
✔ Best model updated and saved to models\single_backbone\EVIT_mfcc_temp.pt

Epoch 3/50


...

Train Loss: 1.0970, Train Acc: 38.79%
Val   Loss: 1.0976, Val   Acc: 33.80%
⚠ No improvement for 8 epoch(s).

Epoch 15/50


Training: 100%|██████████| 774/774 [01:31<00:00,  8.46it/s, acc=38.3, loss=1.1] 


Train Loss: 1.0976, Train Acc: 38.26%
Val   Loss: 1.0948, Val   Acc: 38.54%
⚠ No improvement for 9 epoch(s).

Epoch 16/50


Training: 100%|██████████| 774/774 [01:31<00:00,  8.43it/s, acc=39.2, loss=1.08]


Train Loss: 1.0973, Train Acc: 39.20%
Val   Loss: 1.0979, Val   Acc: 26.89%
⚠ No improvement for 10 epoch(s).
⏹ Early stopping triggered after 16 epochs.

Training complete. Best val accuracy: 47.41%
----- Finished training run 2 -----



TESTING AREA

In [2]:
#Multimodal
folders = [
    r"data\nighttime_data\raw_video",
    r"data\nighttime_data\mfcc_acc"
]
batch_size = 32
resize = (224, 224)

_, _, test_loader = data_loaders.multimodal_loader(
    folders=folders,
    batch_size=batch_size,
    resize=resize,
    return_class_map=False,
    num_classes=6   # or 6
)




In [3]:
from collections import Counter

# Count labels in test_loader
label_counts = Counter()
for batch in test_loader:
    *_, labels = batch  # last element is labels
    label_counts.update(labels.tolist())

print("Samples per class in test set:")
for cls, count in sorted(label_counts.items()):
    print(f"Class {cls}: {count} samples")

Samples per class in test set:
Class 0: 1112 samples
Class 1: 616 samples
Class 2: 1534 samples
Class 3: 1125 samples
Class 4: 688 samples
Class 5: 1530 samples


In [4]:
backbone_configs = [
    {"network": networks.ConvNeXtBaseClassifier, "path": r"models\single_backbone\Night_CNb_video.pt", "nr_classes": 6},  # 6
    {"network": networks.ConvNeXtBaseClassifier,   "path": r"models\single_backbone\Night_CNb_mfcc_acc.pt",   "nr_classes": 6}  # 6
]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

testing_and_metrics_multimodal.test_and_evaluate_multimodal(
    test_loader=test_loader,
    fusion_checkpoint=r"models\preparing_data_models\Night_MM_video_mfcc_acc.pt",
    backbone_configs=backbone_configs,
    output_dir="test_results\Multimodal",   # results saved here
    device=device,
    num_classes=6,               # or 6
    use_amp=True                 # optional: mixed precision evaluation
)


Testing Night_MM_video_mfcc_acc:   0%|          | 0/207 [00:00<?, ?it/s]m:\Coding\Master\masterthesis\testing_and_metrics_multimodal.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Testing Night_MM_video_mfcc_acc: 100%|██████████| 207/207 [00:48<00:00,  4.29it/s]


✔ Results for model 'Night_MM_video_mfcc_acc' saved to: test_results\Multimodal\Night_MM_video_mfcc_acc


In [ ]:
#Single Backbones
models_info = [
    (r"models\single_backbone\.pt", networks.ConvNeXtBaseClassifier),
    (r"models\preparing_data_models\best_model_R34.pt", networks.ResNet34Classifier)
]

image_folder = r"data\time_synchronized_data\cwt_temp"
batch_size = 32
resize = (224, 224)
return_class_map = False

_,_,test_loader = data_loaders.cwt_loader(image_folder, batch_size, resize, return_class_map)

image_folder = r"data\time_synchronized_data\raw_video"
batch_size = 32
resize = (224, 224)
return_class_map = False

_,_,test_loader = data_loaders.general_image_loader(image_folder, batch_size, resize, return_class_map)

image_folder = r"data\time_synchronized_data\mfcc_temp"
batch_size = 32
resize = (224, 224)
return_class_map = False

_,_,test_loader = data_loaders.mfcc_loader(image_folder, batch_size, resize, return_class_map)

testing_and_metrics.test_and_evaluate(
    test_loader=test_loader,
    models_info=models_info,
    output_dir="test_results", ### this is the folder where the results are saved - new folder is generated based on name of model
    device=device,
    num_classes=3
)

Testing CNb_mfcc_temp: 100%|██████████| 104/104 [00:33<00:00,  3.14it/s]


✔ Results for model 'CNb_mfcc_temp' saved to: test_results\CNb_mfcc_temp
